<a href="https://colab.research.google.com/github/yi-juho/AI_LLM_BERT/blob/main/MultiBERT_cordic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K-MHaS (Korean Multi-label Hate Speech Dataset)

## Dataset loading

Loading the K-MHaS dataset from [HuggingFace](https://huggingface.co/datasets/jeanlee/kmhas_korean_hate_speech) and checking meta information (published @COLING2022)


In [ ]:
!pip uninstall datasets fsspec -y
!pip install datasets==3.2.0 fsspec[http]==2024.9.0


In [ ]:
!pip install transformers
!pip install datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("jeanlee/kmhas_korean_hate_speech")

In [ ]:
dataset

In [ ]:
dataset = load_dataset("jeanlee/kmhas_korean_hate_speech", split="test")

In [ ]:
dataset

In [ ]:
dataset.features

In [ ]:
# meta information

print(dataset.info.description)
print(dataset.info.homepage)

In [ ]:
print(dataset.info.citation)

## Data preparation

- Prepare data from train, validation, and test dataset
- Multi-label is converted to multi-label one hot encodding

      class_label:
        names:
          0: origin
          1: physical
          2: politics
          3: profanity
          4: age
          5: gender
          6: race
          7: religion
          8: not_hate_speech

In [ ]:
!pip install transformers
!pip install datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("jeanlee/kmhas_korean_hate_speech")

In [ ]:
!pip install keras_preprocessing


In [ ]:
import tensorflow as tf
import torch

from transformers import BertTokenizer
from transformers import BertForSequenceClassification, AdamW, BertConfig
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from keras_preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, hamming_loss
from sklearn.preprocessing import MultiLabelBinarizer

import pandas as pd
import numpy as np
import random
import time
import datetime
from tqdm import tqdm

import csv
import os



In [ ]:
# load train, validation, and test dataset from HuggingFace

train = load_dataset("jeanlee/kmhas_korean_hate_speech", split="train")
validation = load_dataset("jeanlee/kmhas_korean_hate_speech", split="validation")
test = load_dataset("jeanlee/kmhas_korean_hate_speech", split="test")

In [ ]:
# adding masking (able to remove this step depending on the model)

train_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', train['text']))
validation_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', validation['text']))
test_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', test['text']))

In [ ]:
# convert multi-label to multi-label binary (one hot encoding)
# [8] -> [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]

from sklearn.preprocessing import MultiLabelBinarizer

enc = MultiLabelBinarizer()

def multi_label(example):
    enc_label = enc.fit_transform(example['label'])
    float_arr = np.vstack(enc_label[:]).astype(float)
    update_label = float_arr.tolist()
    return update_label

train_labels = multi_label(train)
validation_labels = multi_label(validation)
test_labels = multi_label(test)

In [ ]:
test_sentences[:5]

In [ ]:
test_labels[:5]

## Prep for Pytorch

In [ ]:
# Tokenizing : bert-base-multilingual-cased

tokenizer = BertTokenizer.from_pretrained('klue/bert-base', do_lower_case=False)

In [ ]:
MAX_LEN = 128

def data_to_tensor (sentences, labels):
  tokenized_texts = [tokenizer.tokenize(sent) for sent in sentences]
  input_ids = [tokenizer.convert_tokens_to_ids(x) for x in tokenized_texts]
  input_ids = pad_sequences(input_ids, maxlen=MAX_LEN, dtype="long", truncating="post", padding="post")

  attention_masks = []

  for seq in input_ids:
      seq_mask = [float(i > 0) for i in seq]
      attention_masks.append(seq_mask)

  tensor_inputs = torch.tensor(input_ids)
  tensor_labels = torch.tensor(labels)
  tensor_masks = torch.tensor(attention_masks)

  return tensor_inputs, tensor_labels, tensor_masks


In [ ]:
train_inputs, train_labels, train_masks = data_to_tensor(train_sentences, train_labels)
validation_inputs, validation_labels, validation_masks = data_to_tensor(validation_sentences, validation_labels)
test_inputs, test_labels, test_masks = data_to_tensor(test_sentences, test_labels)

In [ ]:
batch_size = 32

train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

validation_data = TensorDataset(validation_inputs, validation_masks, validation_labels)
validation_sampler = SequentialSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

test_data = TensorDataset(test_inputs, test_masks, test_labels)
test_sampler = RandomSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

In [ ]:
print('testset size:', len(test_labels))
print('trainset size:', len(train_labels))
print('validset size:', len(validation_labels))

## Multi-BERT model

### GPU setting

In [ ]:
device_name = tf.test.gpu_device_name()
if device_name == '/device:GPU:0':
    print('Found GPU at: {}'.format(device_name))
else:
    raise SystemError('GPU device not found')

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('There are %d GPU(s) available.' % torch.cuda.device_count())
    print('We will use the GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print('No GPU available, using the CPU instead.')

### Model setting

In [ ]:
num_labels = 9

model = BertForSequenceClassification.from_pretrained("klue/bert-base", num_labels=num_labels, problem_type="multi_label_classification")
model.cuda()

In [ ]:
optimizer = AdamW(model.parameters(),
                  lr = 2e-5,
                  eps = 1e-8
                )

# change epochs for improving results (our paper : epochs = 4)
epochs = 4
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0,
                                            num_training_steps = total_steps)

In [ ]:
def format_time(elapsed):
    elapsed_rounded = int(round((elapsed)))
    return str(datetime.timedelta(seconds=elapsed_rounded))  # hh:mm:ss

In [ ]:
def multi_label_metrics(predictions, labels, threshold=0.5):

    # first, apply sigmoid on predictions which are of shape (batch_size, num_labels)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))

    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1

    # finally, compute metrics
    y_true = labels
    accuracy = accuracy_score(y_true, y_pred)
    f1_macro_average = f1_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0)
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0)
    f1_weighted_average = f1_score(y_true=y_true, y_pred=y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro')
    hamming = hamming_loss(y_true, y_pred)

    # return as dictionary
    metrics = {'accuracy': accuracy,
               'f1_macro': f1_macro_average,
               'f1_micro': f1_micro_average,
               'f1_weighted': f1_weighted_average,
               'roc_auc': roc_auc,
               'hamming_loss': hamming}

    return metrics

### Model training

In [ ]:
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

model.zero_grad()
for epoch_i in range(0, epochs):

    # ========================================
    #               Training
    # ========================================

    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')

    t0 = time.time()
    total_loss = 0

    model.train()

    for step, batch in tqdm(enumerate(train_dataloader)):
        if step % 500 == 0 and not step == 0:
            elapsed = format_time(time.time() - t0)
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_input_mask, b_labels = batch

        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask,
                        labels=b_labels)

        loss = outputs[0]
        total_loss += loss.item()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping if it is over a threshold
        optimizer.step()
        scheduler.step()

        model.zero_grad()

    avg_train_loss = total_loss / len(train_dataloader)

    print("")
    print("  Average training loss: {0:.4f}".format(avg_train_loss))
    print("  Training epcoh took: {:}".format(format_time(time.time() - t0)))

print("")
print("Training complete!")

In [ ]:
# ========================================
#               Validation
# ========================================

print("")
print("Running Validation...")

t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []

for batch in validation_dataloader:
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    for b in logits:
        accum_logits.append(list(b))

    for b in label_ids:
        accum_label_ids.append(list(b))

accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)

print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))
print("Hamming Loss: {0:.4f}".format(results['hamming_loss']))
print("Validation took: {:}".format(format_time(time.time() - t0)))

### Evaluation

In [ ]:
# model save

# torch.save(model.state_dict(), path+"BERT_model.pt")

In [ ]:
# load the saved model

path = '/content/model/'
torch.save(model.state_dict(), path+"BERT_multilabel_model.pt")
model.load_state_dict(torch.load(path+"BERT_multilabel_model.pt"))
model.to(device)

In [ ]:
%mkdir model

In [ ]:
t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []
for batch in validation_dataloader:
 batch = tuple(t.to(device) for t in batch)
 b_input_ids, b_input_mask, b_labels = batch
 with torch.no_grad():
  outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
 logits = outputs[0]
 logits = logits.detach().cpu().numpy()
 label_ids = b_labels.to('cpu').numpy()
 for b in logits:
  accum_logits.append(list(b))
 for b in label_ids:
  accum_label_ids.append(list(b))
accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)
print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))

In [ ]:
t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []

for step, batch in tqdm(enumerate(test_dataloader)):
    if step % 100 == 0 and not step == 0:
        elapsed = format_time(time.time() - t0)
        print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(test_dataloader), elapsed))

    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    for b in logits:
        accum_logits.append(list(b))

    for b in label_ids:
        accum_label_ids.append(list(b))

accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)

print("")
print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))
print("Hamming Loss: {0:.4f}".format(results['hamming_loss']))
print("Test took: {:}".format(format_time(time.time() - t0)))

accum_results = []
accum_results.append(list(results.values()))

### Break down evaluation

In [ ]:
for i in range(num_labels):
    ith_label_ids, ith_logits = [], []

    for j, labels in enumerate(accum_label_ids):
        if len(np.where(labels)[0]) == i+1:
            ith_label_ids.append(accum_label_ids[j].tolist())
            ith_logits.append(accum_logits[j].tolist())

    ith_label_ids = np.array(ith_label_ids)
    ith_logits = np.array(ith_logits)

    if len(ith_label_ids) == 0 and len(ith_logits) == 0:
        continue

    results = multi_label_metrics(ith_logits, ith_label_ids)
    accum_results.append(list(results.values()))

    print('# of labels:', i+1)
    print("Accuracy: {0:.4f}".format(results['accuracy']))
    print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro']))
    print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro']))
    print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted']))
    print("ROC-AUC: {0:.4f}".format(results['roc_auc']))
    print("Hamming Loss: {0:.4f}".format(results['hamming_loss']))

    print('\n')

In [ ]:
from transformers import pipeline
pipe = pipeline("text-classification", model=model.cuda(), tokenizer=tokenizer,
device=0, max_length=512,
 return_all_scores=True, function_to_apply='sigmoid')
result = pipe('틀니들은 왜 그렇게 민폐를 끼치냐?')
print(result)
label_dict = {'LABEL_0' : '출신차별', 'LABEL_1' : '외모차별', 'LABEL_2' : '정치성향차별', 'LABEL_3': '혐오욕설', 'LABEL_4': '연령차별', 'LABEL_5': '성차별', 'LABEL_6' : '인종차별', 'LABEL_7': '종교차별', 'LABEL_8': '해당사항없음'}
def prediction(text):
 result = pipe(text)
 return [label_dict[res['label']] for res in result[0] if res['score'] > 0.5]
prediction('틀니들은 왜 그렇게 민폐를 끼치냐?')

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import numpy as np
from torch.nn.functional import sigmoid, softmax
from itertools import chain
import itertools
import pickle


label_names = ['출신차별', '외모차별', '정치성향차별', '혐오욕설', '연령차별', '성차별', '인종차별', '종교차별', '혐오아님']
threshold = 0.1 # 임계값 설정

def predict_labels(text, model, tokenizer, label_names, threshold=0.1):
    # 텍스트를 모델의 입력 형식으로 인코딩
    encoded_dict = tokenizer.encode_plus(
                        text,
                        add_special_tokens = True,
                        max_length = 10,
                        padding = 'max_length',
                        truncation=True,
                        return_attention_mask = True,
                        return_tensors = 'pt',
                        )

    input_ids = encoded_dict['input_ids'].to(device)
    attention_mask = encoded_dict['attention_mask'].to(device)

    # 모델을 사용하여 예측
    with torch.no_grad():
        outputs = model(input_ids, token_type_ids=None, attention_mask=attention_mask)

    # 예측 결과에서 확률 추출
    logits = outputs.logits
    #print(logits)
    probs = sigmoid(logits)
    #print(probs)

    # CPU로 이동 후 numpy 배열로 변환
    probs = probs.detach().cpu().numpy()
    #print(probs)

    # 예측된 레이블 결정
    predicted_labels = [label_names[i] for i in range(len(label_names)) if probs[0][i] >= threshold]

    return predicted_labels

In [ ]:
text = "늙은 천주교 신자들은 다 속물이다"
predicted_labels = predict_labels(text, model, tokenizer, label_names)
print(f"Text: {text} & Predicted labels: {predicted_labels}")

In [ ]:
label_names = ['출신차별', '외모차별', '정치성향차별', '혐오욕설', '연령차별', '성차별', '인종차별', '종교차별', '혐오아님']
threshold = 0.1
text = "못생긴 경상도 여자들은 나가라"
predicted_labels = predict_labels(text, model, tokenizer, label_names)
print(f"Text: {text} -> Predicted labels: {predicted_labels}")

In [ ]:
# VSCode 쪽 코드 (receive_data.py)
outputs_2 = []

import pickle
with open("./attention_scores/attention_scores_1.pkl", "rb") as f:
    outputs_2 = pickle.load(f)

print(outputs_2)

In [ ]:
result = pipe('문재앙')
print(result)

In [ ]:
from transformers import pipeline

# GPU 사용 여부 확인
device = "cpu"

# 모델을 CPU로 이동
model = model.to(device)

pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device, max_length=512, return_all_scores=True)

result = pipe('깜둥이들은 왜 그렇게 민폐를 끼치냐?')
print(result)

In [ ]:
text = "ㅅ발 천주교도들은 너무 말이 많아"
predicted_labels = predict_labels(text, model, tokenizer, label_names)
print(f"Text: {text} & Predicted labels: {predicted_labels}")

In [ ]:
# 파일을 열고 읽은 다음 내용을 출력하는 코드
file_path = '/home/jyhan/HW-output-files/example.txt'  # 읽고자 하는 파일의 경로

try:
    with open(file_path, 'r') as file:  # 파일을 읽기 모드로 열기
        content = file.read()  # 파일의 내용을 전체 읽기
    print(content)  # 읽은 내용 출력
except FileNotFoundError:
    print(f"The file {file_path} does not exist.")  # 파일이 존재하지 않을 경우 에러 메시지 출력

In [ ]:
import pickle

def pickle_to_text(pickle_file_path, text_file_path):
    # 피클 파일 불러오기
    with open(pickle_file_path, "rb") as f:
        data = pickle.load(f)

    # 텍스트 파일로 저장하기
    with open(text_file_path, "w") as f:
        for sublist in data:
            # 각 서브리스트를 반복하고 숫자를 문자열로 변환하여 저장
            for number in sublist:
                f.write(f"{number} ")
            f.write("\n\n\n")  # 각 서브리스트 끝에 줄바꿈 추가

In [ ]:
import torch
import torch.nn.functional as F

def tanh_new(x) :
    result = (F.tanh(x) + 1) /2
    return result

In [ ]:
def encode_number(num):
    """숫자를 7비트 2의 보수 형식으로 인코딩하는 함수, 음수 소수 부분을 고려"""
    num = float(num)  # 입력된 숫자가 문자열이라면, 실수형으로 변환

    # Extract integer and fractional parts
    int_num = int(num)
    frac_num = abs(num - int_num)

    # Round integer part towards zero if num is negative and has a fractional part
    if num < 0:
        if frac_num > 0:
            int_num = int_num - 1  # Round integer part one more negative
            frac_num = 1 - frac_num  # Subtract fractional part from 1 to make it positive

    # Clamp the values to fit within the 7-bit range
    if int_num < -64:
        int_num = -64
    elif int_num > 63:
        int_num = 63

    # Apply 2's complement if the number is negative
    if int_num < 0:
        int_num = (1 << 7) + int_num  # 1 << 7 is 128, representing the range of 7-bit integers

    # Format the number into 7-bit binary
    int_part_bin = format(int_num & 0b1111111, '07b')  # Only the last 7 bits are used

    # Convert the fractional part to binary (13 bits)
    decimal_part_bin = format(int(frac_num * (1 << 13)), '013b')

    # Combine all parts
    encoded = int_part_bin + '_' + decimal_part_bin
    return encoded

In [ ]:
def encode_numbers_input(numbers):
    """리스트의 숫자들을 인코딩하는 함수"""
    return [encode_number(num) for num in numbers]

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import numpy as np
from torch.nn.functional import sigmoid, softmax
from itertools import chain
import itertools

In [ ]:
def encode_number_output(num):
    """숫자를 3비트 정수와 13비트 소수 형식으로 인코딩하는 함수, 값은 0부터 1 사이의 양수"""
    num = float(num)  # 입력된 숫자가 문자열이라면, 실수형으로 변환

    # Clamp the values to fit within the 0 to 1 range
    if num < 0:
        num = 0
    elif num > 1:
        num = 1

    # Extract integer and fractional parts
    int_num = int(num)
    frac_num = num - int_num

    # Convert the fractional part to binary (13 bits)
    decimal_part_bin = format(int(frac_num * (1 << 13)), '013b')

    # Combine all parts
    encoded = '000' + '_' + decimal_part_bin
    return encoded

In [ ]:
def encode_numbers_output(numbers):
    """리스트의 숫자들을 인코딩하는 함수"""
    return [encode_number_output(num) for num in numbers]

In [ ]:
probs = [-3.0061, -4.5561, -3.7781, -4.6988, -1.8251, -4.0483, -4.8075,  4.6085, -5.1693]
encoded_numbers = encode_numbers_input(probs)
print(encoded_numbers)

probs = [0.0472, 0.0104, 0.0224, 0.0090, 0.1388, 0.0172, 0.0081, 0.9901, 0.0057]
encoded_numbers = encode_numbers_output(probs)
print(encoded_numbers)

In [ ]:
probs = [0.0301, 0.0112, 0.0040, 0.7632, 0.0040, 0.0072, 0.0068, 0.8885, 0.0117]
encoded_numbers = encode_numbers_output(probs)
print(encoded_numbers)

In [ ]:
def decode_number(encoded):
    """7비트 2의 보수 형식으로 인코딩된 값을 float으로 디코딩하는 함수"""
    int_part_bin, decimal_part_bin = encoded.split('.')

    # Integer part processing
    if int(int_part_bin, 2) & (1 << 6):  # Negative number (2의 보수)
        int_num = int(int_part_bin, 2) - (1 << 7)
    else:  # Positive number
        int_num = int(int_part_bin, 2)

    # Fractional part processing
    frac_num = int(decimal_part_bin, 2) / (1 << 13)

    # Adjust for negative numbers
    if int_num < 0:
        if frac_num > 0:
            int_num += 1
            frac_num = 1 - frac_num
        result = int_num - frac_num
    else:
        result = int_num + frac_num

    return result

# Test the function with provided values
encoded_values = [
    "0000000.0101010000101",
    "0000000.0010111100000",
    "0000000.0010110110000",
    "0000000.0010001111011",
    "0000000.0010001101110",
    "0000000.0001110010000",
    "0000000.0001100000110",
    "0000000.0000000000010",
    "1111111.1101100010111",
    "1111111.1101011100011"
]

decoded_values = [decode_number(value) for value in encoded_values]
decoded_values

In [ ]:
def decode_number(encoded):
    """7비트 2의 보수 형식으로 인코딩된 값을 float으로 디코딩하는 함수"""
    # 입력 문자열을 반으로 나누어 정수 부분과 소수 부분으로 분리
    mid_index = len(encoded) // 2
    int_part_bin = encoded[:mid_index]
    decimal_part_bin = encoded[mid_index:]

    # 정수 부분 처리
    if int(int_part_bin, 2) & (1 << 6):  # 음수인 경우 (2의 보수)
        int_num = int(int_part_bin, 2) - (1 << 7)
    else:  # 양수인 경우
        int_num = int(int_part_bin, 2)

    # 소수 부분 처리
    frac_num = int(decimal_part_bin, 2) / (1 << 13)

    # 음수 조정
    if int_num < 0:
        if frac_num > 0:
            int_num += 1
            frac_num = 1 - frac_num
        result = int_num - frac_num
    else:
        result = int_num + frac_num

    return result

# 제공된 값으로 함수를 테스트합니다
encoded_values = [
    "00000000101010000101",
    "00000000010111100000",
    "00000000010110110000",
    "00000000010001111011",
    "00000000010001101110",
    "00000000001110010000",
    "00000000001100000110",
    "00000000000000000010",
    "11111111101100010111",
    "11111111101011100011"
]

decoded_values = [decode_number(value) for value in encoded_values]
decoded_values

In [ ]:
def decode_int3_float13(encoded):
    """3비트 정수 + 13비트 소수 형식으로 인코딩된 값을 float으로 디코딩하는 함수"""
    int_part_bin = encoded[:3]
    decimal_part_bin = encoded[3:]

    # 정수 부분 처리 (3비트)
    if int(int_part_bin, 2) & (1 << 2):  # 음수인 경우 (2의 보수)
        int_num = int(int_part_bin, 2) - (1 << 3)
    else:  # 양수인 경우
        int_num = int(int_part_bin, 2)

    # 소수 부분 처리 (13비트)
    frac_num = int(decimal_part_bin, 2) / (1 << 13)

    # 음수 조정
    if int_num < 0:
        if frac_num > 0:
            int_num += 1
            frac_num = 1 - frac_num
        result = int_num - frac_num
    else:
        result = int_num + frac_num

    return result

# 제공된 값으로 함수를 테스트합니다
encoded_values = [
    "0001000101001100",
    "0000011001010100",
    "0000000111011100",
    "0000000111010100",
    "0000000110110100",
    "0000000100000100",
    "0000000011010100",
    "0000000001100100",
    "0000000001010100",
    "0000000001000100"
]

decoded_values = [decode_int3_float13(value) for value in encoded_values]

# 결과 출력
for encoded, decoded in zip(encoded_values, decoded_values):
    print(f"Encoded: {encoded} -> Decoded: {decoded}")

In [ ]:
def format_hex_input(hex_input):
    # 공백을 기준으로 입력값을 분할하여 리스트로 변환
    hex_list = hex_input.split()

    # 리스트를 문자열 형식으로 변환
    formatted_hex_list = "[" + ", ".join(f'"{hex_value}"' for hex_value in hex_list) + "]"

    return formatted_hex_list

# 입력값
hex_input = "06dc 05c4 0494 03c4 039c 02e4 017c 0144 00e4 00ac"
formatted_hex_list = format_hex_input(hex_input)

print(formatted_hex_list)

In [ ]:
def hexa_to_binary(hex_strings):
    binary_strings = [bin(int(hex_string, 16))[2:].zfill(16) for hex_string in hex_strings]
    return binary_strings

def decode_int3_float13(encoded):
    """3비트 정수 + 13비트 소수 형식으로 인코딩된 값을 float으로 디코딩하는 함수"""
    int_part_bin = encoded[:3]
    decimal_part_bin = encoded[3:]

    # 정수 부분 처리 (3비트)
    int_num = int(int_part_bin, 2)

    # 소수 부분 처리 (13비트)
    frac_num = int(decimal_part_bin, 2) / (1 << 13)

    # 결과 계산
    result = int_num + frac_num

    return result

def hexa_to_float(hex_strings):
    binary_strings = hexa_to_binary(hex_strings)
    float_values = [decode_int3_float13(binary_string) for binary_string in binary_strings]
    return float_values

# 새로운 입력값
hex_strings = ["06dc", "05c4", "0494", "03c4", "039c", "02e4", "017c", "0144", "00e4", "00ac"]

float_values = hexa_to_float(hex_strings)

# 변환된 결과 출력
for hex_string, float_value in zip(hex_strings, float_values):
    print(f"{float_value}")

In [ ]:
def format_hex_input(hex_input):
    # 공백을 기준으로 입력값을 분할하여 리스트로 변환
    hex_list = hex_input.split()
    return hex_list

def hexa_to_binary(hex_strings):
    binary_strings = [bin(int(hex_string, 16))[2:].zfill(16) for hex_string in hex_strings]
    return binary_strings

def decode_int3_float13(encoded):
    """3비트 정수 + 13비트 소수 형식으로 인코딩된 값을 float으로 디코딩하는 함수"""
    int_part_bin = encoded[:3]
    decimal_part_bin = encoded[3:]

    # 정수 부분 처리 (3비트)
    int_num = int(int_part_bin, 2)

    # 소수 부분 처리 (13비트)
    frac_num = int(decimal_part_bin, 2) / (1 << 13)

    # 결과 계산
    result = int_num + frac_num

    return result

def hexa_to_float(hex_strings):
    binary_strings = hexa_to_binary(hex_strings)
    float_values = [decode_int3_float13(binary_string) for binary_string in binary_strings]
    return float_values

def convert_hex_input_to_float(hex_input):
    hex_strings = format_hex_input(hex_input)
    float_values = hexa_to_float(hex_strings)
    return float_values

def chunk_floats(float_values, chunk_size=10):
    # float 값을 chunk_size 크기로 분할하고 각 chunk 사이에 두 개의 엔터를 추가
    chunked_result = ""
    for i in range(0, len(float_values), chunk_size):
        chunk = float_values[i:i + chunk_size]
        chunked_result += "\n".join(map(str, chunk)) + "\n\n"
    return chunked_result.strip()

# 입력값
hex_input = """
1404
026c
0234
0214
0184
017c
015c
001c
001c
0004
"""

float_values = convert_hex_input_to_float(hex_input)

# 변환된 결과 출력
chunked_result = chunk_floats(float_values)

print(chunked_result)

In [ ]:
def decode_int3_float13(encoded):
    """3비트 정수 + 13비트 소수 형식으로 인코딩된 값을 float으로 디코딩하는 함수"""
    int_part_bin = encoded[:3]
    decimal_part_bin = encoded[3:]

    # 정수 부분 처리 (3비트)
    int_num = int(int_part_bin, 2)

    # 소수 부분 처리 (13비트)
    frac_num = int(decimal_part_bin, 2) / (1 << 13)

    # 결과 계산
    result = int_num + frac_num

    return result

# 제공된 값으로 함수를 테스트합니다
encoded_values = [
    "0000110101000100",
    "0000100000100100",
    "0000001101111100",
    "0000001011111100",
    "0000000111000100",
    "0000000100000100",
    "0000000001110100",
    "0000000001000100",
    "0000000000111100",
    "0000000000110100"
]

decoded_values = [decode_int3_float13(value) for value in encoded_values]

# 결과 출력
for decoded in decoded_values:
    print(decoded)


In [ ]:
# 새로운 입력값
new_hex_strings = ["0d44", "0824", "037c", "02fc", "01c4", "0104", "0074", "0044", "003c", "0034"]
new_binary_strings = hexa_to_binary(new_hex_strings)

# 변환된 결과 출력
for hex_string, binary_string in zip(new_hex_strings, new_binary_strings):
    print(f"{binary_string}")


In [ ]:
def hexa_to_binary(hex_strings):
    binary_strings = [bin(int(hex_string, 16))[2:].zfill(16) for hex_string in hex_strings]
    return binary_strings

# 새로운 입력값
hex_strings = ["069c", "046c", "045c", "0434", "02fc", "02a4", "026c", "01d4", "00e4"]
binary_strings = hexa_to_binary(hex_strings)

# 변환된 결과 출력
for hex_string, binary_string in zip(hex_strings, binary_strings):
    print(f"{binary_string}")

In [ ]:
# 새로운 입력값
hex_strings = ["045c", "0404", "03b4", "039c", "0374", "034c", "031c", "027c", "022c", "01bc"]
binary_strings = hexa_to_binary(hex_strings)

# 변환된 결과 출력
for hex_string, binary_string in zip(hex_strings, binary_strings):
    print(f"{binary_string}")


In [ ]:
def calculate_error_rates(actual_values, predicted_values):
    """
    여러 개의 실제 값과 예측 값에 대한 오차율을 계산하는 함수.

    매개변수:
    actual_values (list of float): 실제 값 리스트
    predicted_values (list of float): 예측 값 리스트

    반환값:
    list of float: 각 데이터에 대한 오차율 (백분율) 리스트
    """
    if len(actual_values) != len(predicted_values):
        raise ValueError("실제 값 리스트와 예측 값 리스트의 길이는 같아야 합니다.")

    error_rates = []
    for actual, predicted in zip(actual_values, predicted_values):
        if actual == 0:
            error_rate = 0
        else:
            error = actual - predicted
            error_rate = abs((error / actual) * 100)
        error_rates.append(error_rate)

    return error_rates

def calculate_average_error_rate(actual_values, predicted_values):
    """
    여러 개의 실제 값과 예측 값에 대한 평균 오차율을 계산하는 함수.

    매개변수:
    actual_values (list of float): 실제 값 리스트
    predicted_values (list of float): 예측 값 리스트

    반환값:
    float: 평균 오차율 (백분율)
    """
    error_rates = calculate_error_rates(actual_values, predicted_values)
    average_error_rate = sum(error_rates) / len(error_rates)
    return average_error_rate

In [ ]:
# 예제 사용
actual_values = [3.447215, 2.4453104, 1.2259121, 1.2043934, 1.1346276, 0.6330924, 0.4353165, -0.2769869, -0.47311664, -0.6537097]
predicted_values = [3.447215, 2.4453104, 1.2259121, 1.2043934, 1.1346276, 0.6330924, 0.4353165, -0.2769869, -0.47311664, -0.6537097]
average_error_rate = calculate_average_error_rate(actual_values, predicted_values)

print(f"최종 오차율: {average_error_rate}%")

In [ ]:
import time
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

# 10개의 값을 생성
values = np.random.rand(10)

# 실행 시간 측정
start_time = time.time()
softmax_values = softmax(values)
end_time = time.time()

print(f"Softmax output: {softmax_values}")
print(f"Execution time: {end_time - start_time} seconds")

In [ ]:
import timeit
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

# 10개의 값을 생성
values = np.random.rand(10)

# 실행 시간 측정 (timeit 모듈 사용)
execution_time = timeit.timeit(lambda: softmax(values), number=100000)
print(f"Average execution time over 1000 runs: {execution_time / 100000} seconds")


In [ ]:
import time

def my_function():
    # 실행할 코드
    time.sleep(2)  # 예시로 2초 지연

start_time = time.time()
my_function()
end_time = time.time()

print(f"Execution time: {end_time - start_time} seconds")

In [ ]:
import timeit

def my_function():
    # 실행할 코드
    time.sleep(2)  # 예시로 2초 지연

execution_time = timeit.timeit(my_function, number=1)
print(f"Execution time: {execution_time} seconds")


In [ ]:
import time

def measure_time(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"Execution time: {end_time - start_time} seconds")
        return result
    return wrapper

@measure_time
def my_function():
    # 실행할 코드
    time.sleep(2)  # 예시로 2초 지연

my_function()

In [ ]:
text = "현아 성괴"
predicted_labels = predict_labels(text, model, tokenizer, label_names)
print(f"Text: {text} & Predicted labels: {predicted_labels}")